# DQ SRC PERSON

Визуализация последнего прогона `dq-src-person` (логи `DQ_SRC_PERSON`): календарь `_SUCCESS`, period-метрики и профиль snapshot-дня.

Перед запуском: `uv run mobile dq-src-person` или `uv run mobile build-src-person`.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

from mobile.project_paths import PROJECT_ROOT

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

ROOT = PROJECT_ROOT
TAG = "DQ_SRC_PERSON"
BOUNDARY = "dataset_filter"

In [ ]:
from mobile.pipelines.nb.common import checks_by_status, load_dq_dashboard

try:
    dq_logs, latest, meta = load_dq_dashboard(ROOT, tag=TAG, boundary_check=BOUNDARY)
except (FileNotFoundError, ValueError) as exc:
    print(f"Нет DQ-логов для {TAG}: {exc}")
    latest = meta = None
else:
    print("Последний прогон:", meta)
    display(checks_by_status(latest))

In [ ]:
from mobile.pipelines.nb.common import failed_warning_table, metrics_wide_table

if latest is None:
    pass
else:
    display(pd.DataFrame([meta]))
    print("\n--- failed / warning ---")
    display(failed_warning_table(latest))
    print("\n--- все метрики (плоская таблица) ---")
    wide = metrics_wide_table(latest)
    display(wide.head(80))
    if len(wide) > 80:
        print(f"... всего строк metrics: {len(wide)}")

## Визуализации DQ (только метрики из лога)

Все графики — поля `metrics` из JSON-логов `DQ_SRC_PERSON` (без чтения parquet). Прогон в ноутбуке режется по check `dataset_filter`.

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import (
    day_coverage_frame,
    domain_quality_frame,
    identity_fill_frame,
    period_volume_frame,
    render_src_person_dq_distributions,
    render_src_person_dq_overview,
    render_src_person_dq_quality,
    render_src_person_dq_timeseries,
    render_src_person_month_distributions,
)

if latest is None:
    print("Пропуск графиков: нет логов DQ.")
else:
    fig = render_src_person_dq_overview(latest)
    plt.show()
    fig = render_src_person_dq_timeseries(latest)
    plt.show()
    fig = render_src_person_dq_distributions(latest)
    plt.show()
    fig = render_src_person_dq_quality(latest)
    plt.show()
    fig = render_src_person_month_distributions(latest)
    plt.show()
    vol = period_volume_frame(latest)
    if vol.empty:
        vol = day_coverage_frame(latest)
    print(f"Дней в метриках объёма: {len(vol)}")
    print(f"identity_type._fill checks: {len(identity_fill_frame(latest))}")
    print(f"domain quality metrics: {len(domain_quality_frame(latest))}")